# Supplementary analysis: dataset characterization

Descriptive dataset profiles and sensitivity analyses using saved canonical evaluations. The reference is `results/new_characterization_experiments/tables/dataset_experimental_profile.csv`. Seed 42 is loaded and verified against saved folds; it is not refitted in this notebook.


## Reproducibility and execution

This supplement consists of three notebooks in `experiments/`. Run each notebook
from top to bottom with a fresh Python kernel, in this order:

1. `New_Characterization_Experiments.ipynb`: canonical baseline evaluations,
   difficulty scores, score ablation and baseline-dependence experiments.
2. `Default_Tree_Canonical_Strata.ipynb`: default-tree performance and structure
   conditional on the fixed canonical strata.
3. `Dataset_Characterization.ipynb`: descriptive characterization, internal
   consistency checks and sensitivity analyses of saved canonical results.

**Inputs.** Supply the 20 preprocessed CSV files in `data/` at the supplement
root. Exact filenames, target columns and identifier exclusions are defined in
the data-loading cells of the first notebook. Raw data and generated results
are not included in these three source files. Reusing saved results requires
the original CSV bytes, filtered data, software versions and partition hashes;
the notebooks stop on incompatible inputs. Dataset acquisition and preprocessing
must be supplied separately with the publication.

**Environment.** The saved reference uses Python 3.11.5, NumPy 1.23.5,
pandas 2.3.3, SciPy 1.15.3 and scikit-learn 1.3.0. The recorded auxiliary packages are Matplotlib 3.10.8, joblib 1.5.3
and threadpoolctl 3.5.0. IPython and Jupyter are also required. The complete recorded
environment is exported in `results/new_characterization_experiments/metadata.json`.
Saved partition hashes are platform-sensitive; matching package versions alone
does not guarantee compatibility with results created on another platform.

**Protocol.** Use repeated stratified 5-fold cross-validation with 3 repeats,
seed 42, macro-F1 (`zero_division=0`) and sample standard deviation (`ddof=1`).
Preprocessing is fitted within each training fold. Dummy classifiers are
reference baselines and do not contribute to difficulty scores. The canonical
score is 0.70 times normalized predictive error plus 0.30 times normalized
within-model CV variability. Eligibility and exact-tie rules remain defined
in the analysis code; the saved pool contains 17 eligible datasets in strata
of 6, 6 and 5. No hyperparameter search is performed.

**Outputs.** The first two notebooks write under `results/`; the third writes
under `experiments/canonical_alignment/outputs/`. Alternate-seed evaluations
require compatible saved caches or explicit activation of `ALLOW_SEED_FITS`
in the third notebook. Existing source results are checked for consistency.

**Source organization.** Supporting functions are included as readable library
cells at the end of these notebooks. The setup cell loads only the tagged
library definitions into isolated Python namespaces; it does not execute the
other notebooks' analyses or create Python source files. Library cells are
skipped during ordinary top-to-bottom execution because setup already loaded them.

**Interpretation.** Dataset-level summaries are descriptive. Validation using
the same baselines that construct the strata is internal consistency evidence,
not independent predictive validation. Exploratory p-values are unadjusted.
Stored outputs and execution counters have been cleared for this source edition;
run the notebooks to generate tables and figures. The recorded methodology is
preserved; no new scientific results are asserted by this editorial cleanup.

In [ ]:
# Load the documented helper cells from this three-notebook supplement.
from pathlib import Path
import json
import sys
import types

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'experiments' / 'New_Characterization_Experiments.ipynb').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run from the supplement root or its experiments directory.')

def load_supplement_library(module_name, notebook_name, tag, relative_module_path):
    """Load only the explicitly tagged library cell; never run analysis cells."""
    path = PROJECT_ROOT / 'experiments' / notebook_name
    notebook = json.loads(path.read_text(encoding='utf-8'))
    cells = [c for c in notebook['cells']
             if c['cell_type'] == 'code' and tag in c.get('metadata', {}).get('tags', [])]
    if len(cells) != 1:
        raise ValueError(f'Expected one {tag} library cell in {path.name}.')
    for package in ['experiments', 'experiments.canonical_alignment']:
        if package not in sys.modules:
            namespace = types.ModuleType(package)
            namespace.__path__ = []
            sys.modules[package] = namespace
    module = types.ModuleType(module_name)
    module.__file__ = str(PROJECT_ROOT / relative_module_path)
    sys.modules[module_name] = module
    try:
        exec(compile(''.join(cells[0]['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(module_name, None)
        raise
    return module

load_supplement_library(
    'experiments.characterization_persistence', 'New_Characterization_Experiments.ipynb',
    'supplement-persistence', 'experiments/characterization_persistence.py',
)
load_supplement_library(
    'experiments.default_tree_strata', 'Default_Tree_Canonical_Strata.ipynb',
    'supplement-tree', 'experiments/default_tree_strata.py',
)
load_supplement_library(
    'experiments.canonical_alignment.workflow', 'Dataset_Characterization.ipynb',
    'supplement-alignment', 'experiments/canonical_alignment/workflow.py',
)


In [ ]:
# =========================
# 0. Imports and setup
# =========================

from pathlib import Path
import sys
import random
import time
import warnings


import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import entropy, skew, kurtosis, zscore, kruskal, f_oneway
from scipy.spatial.distance import jensenshannon

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, f1_score
from sklearn.feature_selection import mutual_info_classif, chi2

random.seed(42)
np.random.seed(42)

import json, platform, scipy, sklearn
from scipy.stats import spearmanr
from IPython.display import display


In [ ]:
# =========================
# 1. Project root and datasets
# =========================

from pathlib import Path
import sys

def find_project_root(start: Path, marker="experiments/New_Characterization_Experiments.ipynb"):
    for parent in [start] + list(start.parents):
        if (parent / marker).exists():
            return parent
    return start

PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

random.seed(42)
data_path = PROJECT_ROOT / "data"

datasets = {
    "abalone": data_path / "abalone.csv",
    "anneal": data_path / "anneal.csv",
    "arrhythmia": data_path / "arrhythmia.csv",
    "bridges": data_path / "bridges_version1.csv",
    "balance_scale": data_path / "balance_scale.csv",
    "car": data_path / "car.csv",
    "cylinder_bands": data_path / "cylinder_bands.csv",
    "glass": data_path / "glass.csv",
    "heart_disease": data_path / "heart_disease.csv",
    "ionosphere": data_path / "ionosphere.csv",
    "semeion": data_path / "semeion.csv",
    "sonar": data_path / "sonar.csv",
    "tic_tac_toe": data_path / "tic_tac_toe.csv",
    "breast_cancer": data_path / "breast_cancer.csv",
    "wine_red": data_path / "winequality_red.csv",
    "iris": data_path / "iris.csv",
    "haberman": data_path / "haberman.csv",
    "parkinsons": data_path / "parkinsons.csv",
    "banknote": data_path / "banknote_authentication.csv",
    "credit_approval": data_path / "credit_approval.csv",
}

# Edit this list to only include the datasets to analyse
selected_dataset_names = list(datasets.keys())
datasets = {name: path for name, path in datasets.items() if name in selected_dataset_names}

print("Project root:", PROJECT_ROOT)
print("Data path:", data_path)
print("Datasets selected:", list(datasets.keys()))


## 2. Target-column handling

Target columns are resolved explicitly to avoid treating predictors or identifiers as labels.

Target handling:

- `balance_scale`: target is the balance class, usually the **first** column (`class`), not the last numeric attribute.
- `parkinsons`: target is `status`; `name` is an identifier and should be removed from the predictors.
- CV is fixed to the same repeated 5-fold protocol for all eligible datasets. Datasets with fewer than five samples in at least one class are marked as non-comparable instead of being silently evaluated with two folds.

All 20 targets and remaining known IDs are explicitly declared below. Curated bridge/glass CSVs already exclude their IDs. The cylinder-bands cylinder serial number (`feature_001`) is removed here. Target rows with missing labels are removed and counted in the audit.


In [ ]:
# =========================
# 2. Target configuration and loader
# =========================

# Explicit target/ID handling. Values may be column names, integer positions, or
# an ordered list of fallbacks. This prevents accidental target leakage or
# target-column mistakes when the label is not the final column.
TARGET_CONFIG = {'abalone': 'target',
 'anneal': 'target',
 'arrhythmia': 'target',
 'bridges': 'target',
 'balance_scale': ['class'],
 'car': 'target',
 'cylinder_bands': 'target',
 'glass': 'target',
 'heart_disease': '13',
 'ionosphere': '34',
 'semeion': 'target',
 'sonar': '60',
 'tic_tac_toe': '9',
 'breast_cancer': 'target',
 'wine_red': 'target',
 'iris': 'target',
 'haberman': '3',
 'parkinsons': ['status', 'Status'],
 'banknote': '4',
 'credit_approval': '15'}

# Empty lists explicitly mean no remaining known ID columns.
# Bridges/glass IDs were removed during CSV curation.
ID_COLUMNS = {'abalone': [],
 'anneal': [],
 'arrhythmia': [],
 'bridges': [],
 'balance_scale': [],
 'car': [],
 'cylinder_bands': ['feature_001'],
 'glass': [],
 'heart_disease': [],
 'ionosphere': [],
 'semeion': [],
 'sonar': [],
 'tic_tac_toe': [],
 'breast_cancer': [],
 'wine_red': [],
 'iris': [],
 'haberman': [],
 'parkinsons': [['name', 'Name']],
 'banknote': [],
 'credit_approval': []}

MISSING_TOKENS = ["?", "NA", "N/A", "na", "n/a", "", " ", "null", "None"]


def _resolve_column(df, spec, dataset_name, purpose):
    """Resolve a column spec that can be a name, integer position, or fallback list."""
    specs = spec if isinstance(spec, (list, tuple)) else [spec]

    for candidate in specs:
        if isinstance(candidate, int):
            if -len(df.columns) <= candidate < len(df.columns):
                return df.columns[candidate]
        elif candidate in df.columns:
            return candidate

    available = list(df.columns)
    raise ValueError(
        f"Could not resolve {purpose} for {dataset_name}. "
        f"Tried {specs}; available columns: {available}"
    )


def _resolve_existing_columns(df, specs):
    """Return only the configured columns that exist in this concrete CSV."""
    resolved = []
    for spec in specs:
        try:
            col = _resolve_column(df, spec, "dataset", "ID column")
            if col not in resolved:
                resolved.append(col)
        except ValueError as exc:
            raise ValueError("Configured identifier is missing") from exc
    return resolved


def load_dataset(name, path):
    # Load a CSV and split into X and y.
    df = pd.read_csv(path, na_values=MISSING_TOKENS)
    df.columns = [str(c).strip() for c in df.columns]

    if name not in TARGET_CONFIG or name not in ID_COLUMNS:
        raise ValueError(f"Missing explicit target/ID configuration for {name}")
    target_spec = TARGET_CONFIG[name]
    target_col = _resolve_column(df, target_spec, name, "target column")

    id_cols = _resolve_existing_columns(df, ID_COLUMNS[name])
    id_cols = [c for c in id_cols if c != target_col]

    X = df.drop(columns=[target_col] + id_cols).copy()
    y = df[target_col].copy()

    valid_target = ~y.isna()
    X = X.loc[valid_target].reset_index(drop=True)
    y = y.loc[valid_target].reset_index(drop=True)

    y_encoded = LabelEncoder().fit_transform(y.astype(str))

    return {
        "df": df,
        "X": X,
        "y": y_encoded,
        "y_raw": y,
        "target_col": target_col,
        "dropped_id_cols": id_cols,
    }


loaded = {}
target_audit_rows = []

for name, path in datasets.items():
    try:
        obj = load_dataset(name, path)
        loaded[name] = obj
        y_raw = obj["y_raw"]

        target_audit_rows.append({
            "dataset": name,
            "target_col": obj["target_col"],
            "dropped_id_cols": ", ".join(obj["dropped_id_cols"]),
            "n_samples": obj["X"].shape[0],
            "n_features": obj["X"].shape[1],
            "n_classes": pd.Series(obj["y"]).nunique(),
            "min_class_count": pd.Series(obj["y"]).value_counts().min(),
            "class_counts_raw": json.dumps({str(k): int(v) for k, v in y_raw.value_counts().sort_index().items()}),
            "n_missing_targets_removed": int(obj["df"][obj["target_col"]].isna().sum()),
        })

        print(
            f"Loaded {name}: {obj['X'].shape[0]} rows, {obj['X'].shape[1]} features, "
            f"target='{obj['target_col']}', dropped_id_cols={obj['dropped_id_cols']}"
        )
    except Exception as e:
        raise ValueError(f"Could not load {name}: {e}") from e

target_audit_df = pd.DataFrame(target_audit_rows).sort_values("dataset")
target_audit_df


### Target audit

Run this immediately after loading. It makes target mistakes visible before any statistical summaries or baselines are calculated.


In [ ]:
# PAPER-DEFINED: fail before any baseline or statistical analysis.
assert set(datasets) <= set(TARGET_CONFIG), "Every selected dataset needs a target"
assert set(loaded) == set(datasets), "Dataset loading was incomplete"
for name, obj in loaded.items():
    assert obj['X'].columns.is_unique, f"{name}: duplicate predictors"
    assert obj['target_col'] not in obj['X'], f"{name}: target leakage"
    assert len(obj['y']) > 0 and len(np.unique(obj['y'])) >= 2, f"{name}: invalid target"
    assert obj['X'].shape[1] > 0, f"{name}: no predictors"
    assert not obj['y_raw'].isna().any(), f"{name}: missing target after filtering"
    assert not set(obj['dropped_id_cols']) & set(obj['X']), f"{name}: ID leakage"
if 'parkinsons' in loaded:
    p = loaded['parkinsons']
    assert p['target_col'].lower() == 'status' and p['dropped_id_cols'] == ['name']
    assert p['X'].shape == (195, 22) and len(np.unique(p['y'])) == 2
    assert np.bincount(p['y']).min() == 48, "Unexpected Parkinsons class counts"
output_dir = PROJECT_ROOT / 'experiments' / 'canonical_alignment' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
target_audit_df.to_csv(output_dir / 'target_audit.csv', index=False)
display(target_audit_df)


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 3. Dataset-level summary

This gives sample size, dimensionality, class count, missingness, imbalance ratio, and class entropy.


In [ ]:
# =========================
# 3. Dataset-level summary
# =========================

def class_entropy(y):
    counts = np.bincount(y)
    probs = counts / counts.sum()
    return entropy(probs, base=2)


def imbalance_ratio(y):
    counts = np.bincount(y)
    return counts.max() / counts.min() if counts.min() > 0 else np.inf

summary_rows = []
class_distribution_rows = []

for name, obj in loaded.items():
    X, y = obj["X"], obj["y"]
    n_rows, n_features = X.shape
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    counts = pd.Series(y).value_counts(normalize=False).sort_index()
    props = pd.Series(y).value_counts(normalize=True).sort_index()

    summary_rows.append({
        "dataset": name,
        "n_samples": n_rows,
        "n_features": n_features,
        "n_numeric": len(numeric_cols),
        "n_categorical": len(categorical_cols),
        "n_classes": len(np.unique(y)),
        "missing_pct": X.isna().mean().mean() * 100,
        "class_entropy": class_entropy(y),
        "imbalance_ratio": imbalance_ratio(y),
        "samples_per_feature": n_rows / max(n_features, 1),
    })

    for cls in counts.index:
        class_distribution_rows.append({
            "dataset": name,
            "class": cls,
            "count": counts.loc[cls],
            "proportion": props.loc[cls],
        })

summary_df = pd.DataFrame(summary_rows).sort_values("dataset")
class_distribution_df = pd.DataFrame(class_distribution_rows)

output_dir = PROJECT_ROOT / "experiments" / "canonical_alignment" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_dir / "dataset_summary.csv", index=False)
class_distribution_df.to_csv(output_dir / "class_distribution.csv", index=False)

summary_df


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 4. Class balance plots

Important for macro-F1 and for understanding whether a tree may bias itself toward majority classes.


In [ ]:
# ==========================================
# 4. CLASS BALANCE PLOTS (ALL TOGETHER)
# ==========================================

datasets_names = list(loaded.keys())

n_datasets = len(datasets_names)

# Grid size
cols = 4
rows = math.ceil(n_datasets / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 4))

# Flatten axes for easier iteration
axes = axes.flatten()

for idx, name in enumerate(datasets_names):

    temp = class_distribution_df[
        class_distribution_df["dataset"] == name
    ]

    axes[idx].bar(
        temp["class"].astype(str),
        temp["proportion"]
    )

    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel("Class")
    axes[idx].set_ylabel("Prop.")

    axes[idx].tick_params(axis='x', rotation=45)

# Remove unused subplots
for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    "Class Distribution Across Datasets",
    fontsize=18,
    y=1.02
)

plt.tight_layout()
plt.show()


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 5. Missing values analysis

This tells whether missing-value handling genes are likely to matter.


In [ ]:
missing_rows = []
for name, obj in loaded.items():
    X = obj["X"]
    missing_by_feature = X.isna().mean().sort_values(ascending=False) * 100
    for feature, pct in missing_by_feature.items():
        if pct > 0:
            missing_rows.append({"dataset": name, "feature": feature, "missing_pct": pct})

missing_df = pd.DataFrame(missing_rows)
missing_df.to_csv(output_dir / "missing_values_by_feature.csv", index=False)
missing_df.head(30)


In [ ]:
# ==========================================
# MISSING VALUE ANALYSIS (ALL TOGETHER)
# ==========================================
import math
if not missing_df.empty:

    datasets_with_missing = missing_df["dataset"].unique()

    n_datasets = len(datasets_with_missing)

    cols = 2
    rows = math.ceil(n_datasets / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, rows * 4)
    )

    axes = axes.flatten()

    for idx, name in enumerate(datasets_with_missing):

        temp = missing_df[
            missing_df["dataset"] == name
        ].head(15)

        axes[idx].bar(
            temp["feature"].astype(str),
            temp["missing_pct"]
        )

        axes[idx].set_title(name, fontsize=10)
        axes[idx].set_xlabel("Feature")
        axes[idx].set_ylabel("Missing (%)")

        axes[idx].tick_params(
            axis='x',
            rotation=90
        )

    # Remove empty subplots
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(
        "Missing Value Distribution Across Datasets",
        fontsize=18,
        y=1.02
    )

    plt.tight_layout()
    plt.show()


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 6. Numeric descriptive statistics

Mean, median, standard deviation, skewness, kurtosis, and unique values.


In [ ]:
descriptive_rows = []
for name, obj in loaded.items():
    numeric_X = obj["X"].select_dtypes(include=[np.number])
    for feature in numeric_X.columns:
        s = numeric_X[feature].dropna()
        if len(s) == 0:
            continue
        descriptive_rows.append({
            "dataset": name,
            "feature": feature,
            "mean": s.mean(),
            "median": s.median(),
            "std": s.std(),
            "min": s.min(),
            "max": s.max(),
            "skewness": skew(s),
            "kurtosis": kurtosis(s),
            "n_unique": s.nunique(),
        })

descriptive_df = pd.DataFrame(descriptive_rows)
descriptive_df.to_csv(output_dir / "numeric_descriptive_statistics.csv", index=False)
descriptive_df.head(20)


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 7. Low-variance features

Low-variance features may provide little information for splits.


In [ ]:
low_variance_rows = []
for name, obj in loaded.items():
    numeric_X = obj["X"].select_dtypes(include=[np.number])
    for feature in numeric_X.columns:
        s = numeric_X[feature].dropna()
        if len(s) == 0:
            continue
        low_variance_rows.append({
            "dataset": name,
            "feature": feature,
            "variance": s.var(),
            "n_unique": s.nunique(),
        })

low_variance_df = pd.DataFrame(low_variance_rows).sort_values(["dataset", "variance"])
low_variance_df.to_csv(output_dir / "low_variance_features.csv", index=False)
low_variance_df.head(30)


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 8. Outlier analysis

Outliers are estimated using IQR and z-score. They can affect threshold-based splits.


In [ ]:
outlier_rows = []
for name, obj in loaded.items():
    numeric_X = obj["X"].select_dtypes(include=[np.number])
    for feature in numeric_X.columns:
        s = numeric_X[feature].dropna()
        if len(s) < 3:
            continue
        q1, q3 = np.percentile(s, [25, 75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        iqr_outliers = ((s < lower) | (s > upper)).mean() * 100
        z_outliers = (np.abs(zscore(s)) > 3).mean() * 100
        outlier_rows.append({
            "dataset": name,
            "feature": feature,
            "iqr_outlier_pct": iqr_outliers,
            "zscore_outlier_pct": z_outliers,
        })

outlier_df = pd.DataFrame(outlier_rows)
outlier_df.to_csv(output_dir / "outlier_analysis.csv", index=False)
outlier_df.sort_values(["dataset", "iqr_outlier_pct"], ascending=[True, False]).head(30)


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 9. Feature-class separability tests

For numeric features:

- ANOVA checks whether mean values differ across classes.
- Kruskal-Wallis is a non-parametric alternative.

Small p-values suggest that the feature may help separate classes.


In [ ]:
separability_rows = []
for name, obj in loaded.items():
    X, y = obj["X"], obj["y"]
    numeric_X = X.select_dtypes(include=[np.number])
    classes = np.unique(y)
    for feature in numeric_X.columns:
        groups = []
        for cls in classes:
            values = numeric_X.loc[y == cls, feature].dropna().values
            if len(values) > 1:
                groups.append(values)
        if len(groups) < 2:
            continue
        try:
            anova_stat, anova_p = f_oneway(*groups)
        except Exception:
            anova_stat, anova_p = np.nan, np.nan
        try:
            kruskal_stat, kruskal_p = kruskal(*groups)
        except Exception:
            kruskal_stat, kruskal_p = np.nan, np.nan
        separability_rows.append({
            "dataset": name,
            "feature": feature,
            "anova_p": anova_p,
            "kruskal_p": kruskal_p,
            "anova_stat": anova_stat,
            "kruskal_stat": kruskal_stat,
        })

separability_df = pd.DataFrame(separability_rows)
separability_df.to_csv(output_dir / "feature_class_separability_tests.csv", index=False)
separability_df.sort_values(["dataset", "kruskal_p"]).head(30)


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 10. Correlation analysis

Highly correlated numeric features may indicate redundancy.


In [ ]:
correlation_rows = []
for name, obj in loaded.items():
    numeric_X = obj["X"].select_dtypes(include=[np.number])
    if numeric_X.shape[1] < 2:
        continue
    corr = numeric_X.corr(method="spearman").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    high_pairs = upper.stack().reset_index()
    high_pairs.columns = ["feature_1", "feature_2", "abs_spearman_corr"]
    high_pairs = high_pairs.sort_values("abs_spearman_corr", ascending=False)
    for _, row in high_pairs.head(20).iterrows():
        correlation_rows.append({
            "dataset": name,
            "feature_1": row["feature_1"],
            "feature_2": row["feature_2"],
            "abs_spearman_corr": row["abs_spearman_corr"],
        })

correlation_df = pd.DataFrame(correlation_rows)
correlation_df.to_csv(output_dir / "top_feature_correlations.csv", index=False)
correlation_df.head(30)


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 11. Univariate feature importance

Mutual Information and Chi-square scores estimate individual predictive signal per feature.


In [ ]:
def make_encoded_matrix(X):
    X_copy = X.copy()
    numeric_cols = X_copy.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X_copy.columns if c not in numeric_cols]
    parts = []
    feature_names = []

    if numeric_cols:
        X_num = Pipeline([("imputer", SimpleImputer(strategy="median"))]).fit_transform(X_copy[numeric_cols])
        parts.append(X_num)
        feature_names.extend(numeric_cols)

    if categorical_cols:
        X_cat = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]).fit_transform(X_copy[categorical_cols].astype(str))
        parts.append(X_cat)
        feature_names.extend(categorical_cols)

    if not parts:
        return None, []
    return np.hstack(parts), feature_names

importance_rows = []
for name, obj in loaded.items():
    X, y = obj["X"], obj["y"]
    X_encoded, feature_names = make_encoded_matrix(X)
    if X_encoded is None:
        continue
    try:
        mi = mutual_info_classif(X_encoded, y, random_state=42)
    except Exception:
        mi = np.full(X_encoded.shape[1], np.nan)
    try:
        X_nonnegative = MinMaxScaler().fit_transform(X_encoded)
        chi_scores, chi_p = chi2(X_nonnegative, y)
    except Exception:
        chi_scores = np.full(X_encoded.shape[1], np.nan)
        chi_p = np.full(X_encoded.shape[1], np.nan)
    for feature, mi_score, chi_score, p_val in zip(feature_names, mi, chi_scores, chi_p):
        importance_rows.append({
            "dataset": name,
            "feature": feature,
            "mutual_information": mi_score,
            "chi2_score": chi_score,
            "chi2_p": p_val,
        })

importance_df = pd.DataFrame(importance_rows)
importance_df.to_csv(output_dir / "univariate_feature_importance.csv", index=False)
importance_df.sort_values(["dataset", "mutual_information"], ascending=[True, False]).head(30)


## 12. Baseline performance and dataset difficulty dimensions

Baselines use macro-F1 under the same repeated stratified 5-fold protocol.

The dataset characterization is explicitly separated into three dimensions:

1. **Predictive difficulty**: low baseline macro-F1 plus high within-model CV instability.
2. **Strategy sensitivity**: disagreement between baseline algorithms.
3. **Computational burden**: runtime and tree-size/complexity indicators.

The primary Easy/Intermediate/Hard label is based only on predictive difficulty. Model spread, tree size, and runtime remain available as complementary experimental-profile descriptors, and never enter the main difficulty category.


In [ ]:
# Load the audited saved execution; never refit an implicit seed 42.
from experiments.canonical_alignment import workflow as alignment
alignment_context = alignment.prepare()
output_dir = alignment.safe_path(output_dir)
assert output_dir == alignment.OUT.resolve(), 'Exports must use the isolated output directory'
from experiments.characterization_persistence import data_digest
assert set(loaded) == set(alignment_context['ns']['loaded'])
for dataset_name, dataset_obj in loaded.items():
    assert data_digest(dataset_obj) == alignment_context['metadata']['datasets'][dataset_name]['filtered_data_sha256']
for name in ['NON_DUMMY_BASELINE_MODELS', 'TREE_BASELINE_MODELS', 'DUMMY_REFERENCE_MODELS',
             'MODEL_CONFIGS', 'CV_RANDOM_STATE', 'CV_N_SPLITS', 'CV_N_REPEATS', 'N_JOBS',
             'successful_rows', 'aggregate_baselines', 'minmax_normalize_on_pool',
             'assign_predictive_difficulty_groups']:
    globals()[name] = alignment_context['ns'][name]
baseline_df = alignment_context['baseline'].copy(deep=True)
cv_audit_df = alignment_context['audit'].copy(deep=True)
baseline_characteristics_df = aggregate_baselines(baseline_df, cv_audit_df)
print('Canonical source:', alignment_context['provenance']['canonical_source'])
print('SHA-256:', alignment_context['provenance']['canonical_sha256'])
print('Saved baseline evaluation loaded; zero canonical models fitted.')
display(baseline_characteristics_df)


In [ ]:
plot_df = baseline_df.dropna(subset=["f1_macro_mean"])
for model_name in plot_df["model"].unique():
    temp = plot_df[plot_df["model"] == model_name].sort_values("f1_macro_mean", ascending=False)
    plt.figure(figsize=(10, 5))
    plt.bar(temp["dataset"], temp["f1_macro_mean"])
    plt.title(f"Baseline macro-F1 by dataset - {model_name}")
    plt.xlabel("Dataset")
    plt.ylabel("Mean macro-F1")
    plt.xticks(rotation=90)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()


In [ ]:
# PAPER-DEFINED computational inputs, calculated once.
computational_source_df = baseline_df.loc[successful_rows(baseline_df)
    & baseline_df.model.isin(NON_DUMMY_BASELINE_MODELS)]
runtime_df = computational_source_df.groupby('dataset').agg(
    baseline_runtime_seconds_mean=('runtime_seconds','mean')).reset_index()
tree_complexity_df = computational_source_df.loc[
    computational_source_df.model.isin(TREE_BASELINE_MODELS)].groupby('dataset').agg(
    mean_tree_nodes=('mean_tree_nodes','mean'), mean_tree_depth=('mean_tree_depth','mean'),
    mean_tree_leaves=('mean_tree_leaves','mean')).reset_index()
computational_characteristics_df = pd.DataFrame({'dataset': list(loaded)}).merge(
    runtime_df, on='dataset', how='left').merge(tree_complexity_df, on='dataset', how='left')
display(computational_characteristics_df)


In [ ]:
# Canonical profile loaded with original, unrounded components and descriptors.
dataset_profile_df = alignment_context['profile'].copy(deep=True)
alignment.assert_profile(dataset_profile_df, alignment_context['reference'])
SCORE_TIE_ATOL = 0.0
# Normalization/ranking helpers are the audited canonical source definitions.
# Monotonicity audit
# ============================================================
#
# The paper verifies:
# mean(F)_Easy > mean(F)_Intermediate > mean(F)_Hard
#
# Mean is used because predictive error E_d is itself derived from
# mean baseline macro-F1.


ordered_labels = ["easy", "intermediate", "hard"]

predictive_monotonicity_df = (
    dataset_profile_df[
        dataset_profile_df["predictive_difficulty_label"].isin(
            ordered_labels
        )
    ]
    .groupby("predictive_difficulty_label", observed=True)
    .agg(
        n_datasets=("dataset", "nunique"),
        mean_baseline_macro_f1=("baseline_macro_f1_mean", "mean"),
        min_baseline_macro_f1=("baseline_macro_f1_mean", "min"),
        max_baseline_macro_f1=("baseline_macro_f1_mean", "max"),
        mean_cv_variability=("cv_variability", "mean"),
        mean_strategy_sensitivity=("strategy_sensitivity", "mean"),
        mean_computational_burden=("computational_burden_score", "mean"),
    )
    .reindex(ordered_labels)
    .reset_index()
)

group_means = (
    predictive_monotonicity_df
    .set_index("predictive_difficulty_label")["mean_baseline_macro_f1"]
)

is_monotonic_f1 = (
    group_means.notna().all()
    and group_means["easy"] > group_means["intermediate"]
    and group_means["intermediate"] > group_means["hard"]
)

print(
    "Monotonic predictive difficulty "
    "(Easy > Intermediate > Hard):",
    bool(is_monotonic_f1),
)


# ============================================================
# 6. Export and display
# ============================================================


dataset_profile_df.to_csv(
    output_dir / "dataset_experimental_profile.csv",
    index=False,
)

predictive_monotonicity_df.to_csv(
    output_dir / "predictive_difficulty_monotonicity_check.csv",
    index=False,
)


display(predictive_monotonicity_df)

ranking_columns = [
    "dataset",
    "baseline_macro_f1_mean",       # F_d
    "predictive_error_score",       # z(E_d)
    "cv_variability_score",         # z(V_d)
    "predictive_difficulty_score",  # D_pred_d
    "predictive_difficulty_rank",
    "predictive_difficulty_label",
    "strategy_sensitivity",         # S_d
    "strategy_sensitivity_score",
]

for optional_col in [
    "baseline_runtime_seconds_mean",
    "mean_tree_nodes",
    "mean_tree_depth",
    "mean_tree_leaves",
    "computational_burden_score",
]:
    if optional_col in dataset_profile_df.columns:
        ranking_columns.append(optional_col)


display(
    dataset_profile_df[ranking_columns]
    .sort_values(
        ["predictive_difficulty_rank", "dataset"],
        na_position="last",
    )
)


## Paper-defined validation after fixed subset assignment
The dataset is the unit of analysis. Tree medians are the primary validation;
logistic regression is complementary. These checks describe the same baselines
used to construct the strata and are internal validation, not independent evidence.
Variability and sensitivity need not be monotonic and never reclassify datasets.


In [ ]:
canonical_snapshot_df = dataset_profile_df.copy(deep=True)
def distribution_summary(values):
    values = pd.Series(values).dropna()
    q1, q3 = values.quantile([.25,.75])
    return dict(n_datasets=len(values), median=values.median(), Q1=q1, Q3=q3, IQR=q3-q1)

validation_source = baseline_df.loc[successful_rows(baseline_df)
    & baseline_df.model.isin(NON_DUMMY_BASELINE_MODELS)].merge(
    dataset_profile_df[['dataset','ranking_eligible','predictive_difficulty_label']],
    on='dataset', validate='many_to_one')
validation_source = validation_source.loc[validation_source.ranking_eligible]
validation_rows = []
for model in NON_DUMMY_BASELINE_MODELS:
    for subset in ordered_labels:
        values = validation_source.loc[(validation_source.model == model)
            & (validation_source.predictive_difficulty_label == subset), 'f1_macro_mean']
        validation_rows.append(dict(model=model, subset=subset, **distribution_summary(values)))
subset_predictive_validation_df = pd.DataFrame(validation_rows)
tree_validation_rows=[]
for model in NON_DUMMY_BASELINE_MODELS:
    medians = subset_predictive_validation_df.query('model == @model').set_index('subset')['median']
    tree_validation_rows.append(dict(model=model, primary_tree_validation=model in TREE_BASELINE_MODELS,
        median_ordering_pass=bool(medians['easy'] > medians['intermediate'] > medians['hard'])))
subset_median_ordering_df = pd.DataFrame(tree_validation_rows)
subset_predictive_validation_df.to_csv(output_dir / 'subset_predictive_validation.csv', index=False)
subset_median_ordering_df.to_csv(output_dir / 'subset_median_ordering.csv', index=False)
display(subset_predictive_validation_df)
display(subset_median_ordering_df)

descriptive_rows=[]
for subset in ordered_labels:
    group=dataset_profile_df.loc[dataset_profile_df.predictive_difficulty_label.eq(subset)]
    for metric in ['baseline_macro_f1_mean','cv_variability','strategy_sensitivity',
                   'baseline_runtime_seconds_mean','mean_tree_nodes']:
        descriptive_rows.append(dict(subset=subset, metric=metric, **distribution_summary(group[metric])))
subset_descriptive_statistics_df=pd.DataFrame(descriptive_rows)
subset_descriptive_statistics_df.to_csv(output_dir / 'subset_descriptive_statistics.csv', index=False)
stability_sensitivity_profiles_df=subset_descriptive_statistics_df.loc[
    subset_descriptive_statistics_df.metric.isin(['cv_variability','strategy_sensitivity'])].copy()
stability_sensitivity_profiles_df.to_csv(output_dir / 'subset_stability_sensitivity_profiles.csv', index=False)
display(subset_descriptive_statistics_df)


## Supporting dataset-level Spearman and feature separability
These exploratory associations do not affect labels. Each correlation uses one
row per eligible dataset, with pairwise finite observations. P-values are unadjusted;
multiple testing and the small benchmark pool preclude confirmatory interpretation.
Separability proportions use valid p-values for each test as their denominator;
the counts of attempted and valid tests are exported separately.


In [ ]:
association_source=dataset_profile_df.merge(summary_df, on='dataset', validate='one_to_one')
association_source=association_source.loc[association_source.ranking_eligible]
correlation_rows_dataset=[]
for indicator in ['predictive_difficulty_score','predictive_error_score','cv_variability_score',
                  'strategy_sensitivity_score','computational_burden_score']:
    for property_name in ['n_features','samples_per_feature','missing_pct','class_entropy',
                          'imbalance_ratio','n_samples','n_classes']:
        pair=association_source[[indicator,property_name]].replace([np.inf,-np.inf],np.nan).dropna()
        valid=len(pair)>=3 and pair.nunique().min()>1
        rho,pvalue=spearmanr(pair[indicator],pair[property_name]) if valid else (np.nan,np.nan)
        correlation_rows_dataset.append(dict(indicator=indicator, property=property_name,
            rho=rho, p_value=pvalue, n_datasets=len(pair), status='ok' if valid else 'insufficient or constant'))
dataset_level_spearman_df=pd.DataFrame(correlation_rows_dataset)
dataset_level_spearman_df.to_csv(output_dir / 'dataset_level_spearman_correlations.csv',index=False)
separability_summary_rows=[]
for name in loaded:
    tests=separability_df.loc[separability_df.dataset.eq(name)]
    row=dict(dataset=name,n_numeric_features_tested=len(tests))
    for test in ['anova','kruskal']:
        pvalues=tests[test+'_p'].dropna()
        row['n_valid_'+test+'_tests']=len(pvalues)
        row['proportion_'+test+'_p_lt_0_05']=(pvalues<.05).mean() if len(pvalues) else np.nan
    separability_summary_rows.append(row)
feature_separability_dataset_summary_df=pd.DataFrame(separability_summary_rows)
feature_separability_dataset_summary_df.to_csv(output_dir / 'feature_separability_dataset_summary.csv',index=False)
display(dataset_level_spearman_df)
display(feature_separability_dataset_summary_df)


## Robustness analysis — not used for canonical label assignment
Alternative weights reuse canonical normalization and the same exact-tie ranking
and equal-group procedure. All alternatives live in separate DataFrames.


In [ ]:
# Change only alpha; the canonical E, V and normalized components remain fixed.
alignment.assert_profile(canonical_snapshot_df, alignment_context['reference'])
weight_sensitivity_summary_df, weight_sensitivity_dataset_df = alignment.weights(alignment_context)
display(weight_sensitivity_summary_df)
display(weight_sensitivity_dataset_df.loc[weight_sensitivity_dataset_df.subset_changed])
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(weight_sensitivity_summary_df.alpha, weight_sensitivity_summary_df.n_subset_changes, marker='o')
ax.set(xlabel='Predictive error weight (alpha)', ylabel='Datasets changing stratum')
fig.tight_layout()
plt.show()


## Robustness analysis: cross-validation seed

Only the partition seed changes for seeds 7, 21, 84 and 123: identical data, targets, preprocessing, four non-dummy models, fixed estimator random states, macro-F1 and 5-fold × 3-repeat CV in the canonical software environment. Historical seed results were rejected: their logs report sklearn 1.8, whereas the canonical run uses sklearn 1.3. The new evaluations have per-model configuration, data and partition fingerprints.

Seed **42 is the saved canonical evaluation**, loaded and checked algebraically from its complete fold vectors. This is not a new training reproduction. The other four seeds are independently evaluated and cached. Every seed uses the same 17-dataset pool; incomplete or mixed-protocol evaluations fail before normalization. E and V are normalized separately within each seed over all 17 datasets, with the original exact-score tie breakers and 6/6/5 partition. Alternative seeds never redefine canonical labels.


In [ ]:
# Set True only when intentionally computing missing alternate-seed evaluations.
ALLOW_SEED_FITS = False
# Read/recompute-only default: missing compatible caches fail visibly.
ROBUSTNESS_CV_SEEDS = alignment.SEEDS
SEED_REPRODUCTION_ATOL = 1e-12
seed_reference_df = alignment_context['reference'].sort_values('predictive_difficulty_rank').copy(deep=True)
seed_output_dir = output_dir
print('Seed 42: loaded_not_refitted. Alternative seeds: verified canonical-environment caches.')


### Load verified evaluations and recompute the five seed profiles

The workflow checks all 68 model evaluations per seed. Recalculation includes per-seed aggregates and normalization, rankings, strata, boundary crossings, and dataset-level summaries. Across-seed statistics contain four alternate evaluations plus the saved canonical reference.


In [ ]:
alignment.assert_profile(dataset_profile_df, alignment_context['reference'])
try:
    (seed_robustness_summary_df, seed_robustness_comparison_vs_canonical_df,
     seed_robustness_dataset_summary_df, seed_robustness_dataset_scores_df) = alignment.seeds(
         alignment_context, allow_fit=ALLOW_SEED_FITS)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        'A verified alternate-seed cache is missing. Supply the original seed_cache '
        'directory or set ALLOW_SEED_FITS = True in the preceding setup cell '
        'to compute missing evaluations in the canonical environment.'
    ) from None
display(seed_robustness_summary_df)


### Robustness assertions

These are saved-reference consistency checks, not claims of seed-42 training reproduction. All five profiles must have 17 distinct datasets and strata of 6/6/5. Source hashes must still match, and the loaded profile must remain identical to the reference.


In [ ]:
alignment.assert_profile(dataset_profile_df, alignment_context['reference'])
seed42 = seed_robustness_dataset_scores_df.loc[seed_robustness_dataset_scores_df.seed.eq(42)]
alignment.assert_profile(seed42, seed_reference_df)
assert set(seed_robustness_dataset_scores_df.seed) == set(ROBUSTNESS_CV_SEEDS)
for seed_value, frame in seed_robustness_dataset_scores_df.groupby('seed'):
    alignment.keyed(frame, alignment_context['pool'])
    assert frame.normalization_pool_size.eq(17).all()
    assert frame.predictive_difficulty_label.value_counts().reindex(alignment.GROUPS).tolist() == [6,6,5]
assert seed_robustness_summary_df.query('seed == 42').reference_status.eq('loaded_not_refitted').all()
assert dataset_profile_df.equals(canonical_snapshot_df)
protected_file_count = alignment.verify_protected()
seed_checks_df = pd.DataFrame([
    {'check':'Saved seed 42 matches canonical values (no new training)', 'passed':True},
    {'check':'All five seeds use the complete fixed pool and 6/6/5 strata', 'passed':True},
    {'check':'Per-model data, configuration, environment and split fingerprints validated', 'passed':True},
    {'check':'Canonical profile and protected artifacts unchanged', 'passed':True}])
alignment.csv(seed_checks_df, 'seed_robustness_assertions.csv')
display(seed_checks_df)


### Rank trajectories, boundary crossings and score dispersion

The three seed figures and the weight figure are exported as PNG and PDF. The seed-42 tick explicitly denotes a saved reference. Changes are evaluated by dataset name; rank changes and boundary crossings are reported separately.


In [ ]:
alignment.figures(weight_sensitivity_summary_df, seed_robustness_summary_df,
    seed_robustness_comparison_vs_canonical_df, seed_robustness_dataset_summary_df,
    seed_robustness_dataset_scores_df)
from IPython.display import Image
for figure_name in ['weight_sensitivity', 'cv_seed_rank_stability',
                    'cv_seed_label_stability', 'cv_seed_predictive_difficulty_stability']:
    display(Image(filename=str(output_dir / (figure_name + '.png'))))


In [ ]:
alternatives = seed_robustness_summary_df.loc[seed_robustness_summary_df.seed.ne(42)]
print('Seed 42 is loaded; it was not refitted in this analysis.')
print(f'Alternative-seed rank correlation: min={alternatives.spearman_rank_correlation.min():.6f}; mean={alternatives.spearman_rank_correlation.mean():.6f}')
print(f'Maximum label changes: {alternatives.n_subset_changes.max()}/17')
changed_datasets = seed_robustness_dataset_summary_df.loc[seed_robustness_dataset_summary_df.n_label_changes.gt(0), 'dataset']
print('Datasets crossing a boundary: ' + ', '.join(changed_datasets))
display(seed_robustness_dataset_summary_df)
display(seed_robustness_comparison_vs_canonical_df)
alignment.assert_profile(dataset_profile_df, alignment_context['reference'])


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 14. Dataset similarity based on class distributions

Uses Jensen-Shannon distance between class distributions. This is a rough comparison because datasets may have different numbers of classes.


In [ ]:
# EXPLORATORY: frequency shapes, independent of semantic class labels.
names = list(loaded)
max_classes = max(len(np.unique(loaded[n]['y'])) for n in names)
def class_frequency_profile(y, width):
    proportions = pd.Series(y).value_counts(normalize=True).to_numpy()
    return np.pad(np.sort(proportions)[::-1], (0, width-len(proportions)))
class_vectors = {n: class_frequency_profile(loaded[n]['y'], max_classes) for n in names}
for n in names:
    y = loaded[n]['y']
    permuted = np.max(y) - y
    assert np.array_equal(class_vectors[n], class_frequency_profile(permuted, max_classes))
    for other in names:
        assert jensenshannon(class_vectors[n], class_vectors[other]) == jensenshannon(
            class_frequency_profile(permuted, max_classes), class_vectors[other])
similarity_matrix = pd.DataFrame([[jensenshannon(class_vectors[a], class_vectors[b])
    for b in names] for a in names], index=names, columns=names)
similarity_matrix.to_csv(output_dir / 'class_distribution_js_distance.csv')
print('PASS: class-label permutation leaves frequency profiles and distances unchanged')
display(similarity_matrix)


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix.astype(float), aspect="auto")
plt.colorbar(label="Jensen-Shannon distance")
plt.xticks(range(len(names)), names, rotation=90)
plt.yticks(range(len(names)), names)
plt.title("Dataset distance based on class distributions")
plt.tight_layout()
plt.show()


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

## 15. Dataset interpretation table

This final table merges the most useful indicators for discussion.
`possible_issue` thresholds are heuristic interpretation aids only, outside the paper-defined classification.


In [ ]:
tree_baselines = baseline_df[
    baseline_df["model"].isin(["decision_tree_default", "decision_tree_balanced"])
].pivot_table(
    index="dataset",
    columns="model",
    values=["f1_macro_mean", "f1_macro_std"],
    aggfunc="first",
)

tree_baselines.columns = [
    f"{model}_{metric}".replace("decision_tree_", "tree_")
    for metric, model in tree_baselines.columns
]
tree_baselines = tree_baselines.reset_index()

# Read canonical quantities directly; heuristic flags below cannot change them.
final_interpretation_df = summary_df.merge(tree_baselines, on='dataset', how='left').merge(
    dataset_profile_df, on='dataset', how='left', validate='one_to_one')

# Backward-compatible names for the default tree baseline.
if "tree_default_f1_macro_mean" in final_interpretation_df.columns:
    final_interpretation_df["baseline_tree_f1_mean"] = final_interpretation_df["tree_default_f1_macro_mean"]
if "tree_default_f1_macro_std" in final_interpretation_df.columns:
    final_interpretation_df["baseline_tree_f1_std"] = final_interpretation_df["tree_default_f1_macro_std"]

# Backward-compatible difficulty column, now explicitly predictive rather than
# broad experimental challenge.
final_interpretation_df["tree_difficulty_revised"] = final_interpretation_df["predictive_difficulty_label"]

final_interpretation_df["possible_issue"] = ""
final_interpretation_df.loc[final_interpretation_df["imbalance_ratio"] > 2, "possible_issue"] += "class imbalance; "
final_interpretation_df.loc[final_interpretation_df["missing_pct"] > 5, "possible_issue"] += "missing values; "
final_interpretation_df.loc[final_interpretation_df["samples_per_feature"] < 10, "possible_issue"] += "low samples/features ratio; "
final_interpretation_df.loc[final_interpretation_df["comparable_cv"] == False, "possible_issue"] += "not eligible for fixed 5-fold CV; "
final_interpretation_df.loc[final_interpretation_df["cv_variability"] > 0.10, "possible_issue"] += "high within-model CV variability; "
final_interpretation_df.loc[final_interpretation_df["baseline_macro_f1_spread"] > 0.20, "possible_issue"] += "large strategy sensitivity; "
final_interpretation_df.loc[final_interpretation_df["computational_burden_score"] > 0.66, "possible_issue"] += "high computational burden; "

final_interpretation_df["possible_issue"] = final_interpretation_df["possible_issue"].replace("", "none obvious")

final_interpretation_df.to_csv(output_dir / "thesis_dataset_interpretation_summary.csv", index=False)
final_interpretation_df.sort_values(["predictive_difficulty_label", "baseline_macro_f1_mean"], ascending=[True, False])


**EXPLORATORY / SUPPORTING DESCRIPTORS — never used to assign labels.**

Justify why each dataset may affect the GA differently:

- The primary `predictive_difficulty_label` is driven by baseline macro-F1 and within-model CV instability.
- `strategy_sensitivity` reports algorithm-to-algorithm disagreement and never changes the main difficulty label.
- `computational_burden_score` reports runtime/tree-size pressure and is also separate from the main difficulty label.
- Dataset imbalance can affect macro-F1 and make some datasets harder.
- Missing values make missing-value handling genes more relevant.
- Strong univariate feature importance may indicate that simple trees can already perform well.
- High `cv_variability` means fold-to-fold instability within models is large.
- High `baseline_macro_f1_spread` means algorithms differ substantially on the dataset.
- Dataset differences help explain why SDD alone may encourage equal behavior without necessarily improving absolute F1.


In [ ]:
# Full-precision canonical table: no publication rounding enters analyses.
canonical_article_full_precision_df = dataset_profile_df.loc[dataset_profile_df.ranking_eligible,
    ['dataset', 'baseline_macro_f1_mean', 'predictive_error', 'cv_variability',
     'predictive_error_score', 'cv_variability_score', 'predictive_difficulty_score',
     'predictive_difficulty_rank', 'predictive_difficulty_label']].sort_values('predictive_difficulty_rank')
alignment.csv(canonical_article_full_precision_df, 'canonical_article_full_precision.csv')
# ============================================================
# Table for the article: predictive-difficulty ranking
# ============================================================

article_ranking_df = (
    dataset_profile_df.loc[
        dataset_profile_df["predictive_difficulty_label"].isin(
            ["easy", "intermediate", "hard"]
        ),
        [
            "dataset",
            "baseline_macro_f1_mean",
            "predictive_error_score",
            "cv_variability_score",
            "predictive_difficulty_score",
            "predictive_difficulty_label",
        ],
    ]
    .loc[dataset_profile_df.loc[dataset_profile_df.ranking_eligible].sort_values("predictive_difficulty_rank").index]
    .copy()
)

# Names formatted for the paper
dataset_display_names = {
    "banknote": "Banknote Authentication",
    "breast_cancer": "Breast Cancer",
    "car": "Car Evaluation",
    "credit_approval": "Credit Approval",
    "cylinder_bands": "Cylinder Bands",
    "heart_disease": "Heart Disease",
    "tic_tac_toe": "Tic-Tac-Toe",
    "wine_red": "Wine Quality Red",
    "anneal": "Anneal",
    "balance_scale": "Balance Scale",
    "glass": "Glass",
    "haberman": "Haberman",
    "ionosphere": "Ionosphere",
    "iris": "Iris",
    "semeion": "Semeion",
    "sonar": "Sonar",
    "parkinsons": "Parkinsons",
}

article_ranking_df["dataset"] = (
    article_ranking_df["dataset"]
    .map(dataset_display_names)
    .fillna(article_ranking_df["dataset"])
)

article_ranking_df["predictive_difficulty_label"] = (
    article_ranking_df["predictive_difficulty_label"]
    .str.capitalize()
)

article_ranking_df = article_ranking_df.rename(
    columns={
        "dataset": "Dataset",
        "baseline_macro_f1_mean": "Mean baseline macro-F1",
        "predictive_error_score": "z(E_d)",
        "cv_variability_score": "z(V_d)",
        "predictive_difficulty_score": "D_pred_d",
        "predictive_difficulty_label": "Final subset",
    }
)

numeric_columns = [
    "Mean baseline macro-F1",
    "z(E_d)",
    "z(V_d)",
    "D_pred_d",
]

article_ranking_df[numeric_columns] = (
    article_ranking_df[numeric_columns].round(4)
)

display(article_ranking_df)

latex_table = article_ranking_df.to_latex(
    index=False,
    escape=False,
    column_format="lrrrrl",
    caption=(
        "Predictive-difficulty ranking of the eligible datasets. "
        "The score combines normalized predictive error and "
        "cross-validation variability with weights of 0.70 and 0.30, "
        "respectively. Lower values indicate more favorable predictive behavior."
    ),
    label="tab:predictive_difficulty_ranking",
    position="t",
    float_format="%.4f",
)

latex_path = output_dir / "predictive_difficulty_ranking_table.tex"
latex_path.write_text(latex_table, encoding="utf-8")

print(latex_table)
print(f"\nSaved to: {latex_path}")

alignment.csv(article_ranking_df, 'canonical_article_display.csv')


In [ ]:
# Read-only computational-burden display; uses the canonical values.
display(
    dataset_profile_df[
        [
            "dataset",
            "baseline_runtime_seconds_mean",
            "mean_tree_nodes",
            "mean_tree_depth",
            "mean_tree_leaves",
            "runtime_burden_score",
            "tree_size_burden_score",
            "computational_burden_score",
        ]
    ]
    .sort_values(
        "computational_burden_score",
        ascending=False,
    )
)


In [ ]:
# ============================================================
# Multidimensional dataset-profile heatmap
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


required_columns = [
    "dataset",
    "predictive_difficulty_rank",
    "predictive_difficulty_score",
    "strategy_sensitivity_score",
    "computational_burden_score",
    "predictive_difficulty_label",
]

missing_columns = [
    column
    for column in required_columns
    if column not in dataset_profile_df.columns
]

if missing_columns:
    raise KeyError(
        "The following columns are required to generate the profile figure: "
        f"{missing_columns}"
    )


# Keep only datasets included in the final ranking
profile_plot_df = dataset_profile_df.loc[
    dataset_profile_df["predictive_difficulty_label"].isin(
        ["easy", "intermediate", "hard"]
    ),
    required_columns,
].copy()


# Explicit order of the subsets
label_order = {
    "easy": 0,
    "intermediate": 1,
    "hard": 2,
}

profile_plot_df["label_order"] = (
    profile_plot_df["predictive_difficulty_label"]
    .map(label_order)
)


# Sort first by subset and then by predictive-difficulty score
profile_plot_df = (
    profile_plot_df
    .sort_values(
        [
            "label_order",
            "predictive_difficulty_rank",
        ],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)


# Dataset names formatted for the paper
dataset_display_names = {
    "banknote": "Banknote Authentication",
    "breast_cancer": "Breast Cancer",
    "car": "Car Evaluation",
    "credit_approval": "Credit Approval",
    "cylinder_bands": "Cylinder Bands",
    "heart_disease": "Heart Disease",
    "tic_tac_toe": "Tic-Tac-Toe",
    "wine_red": "Wine Quality Red",
    "balance_scale": "Balance Scale",
    "anneal": "Anneal",
    "glass": "Glass",
    "haberman": "Haberman",
    "ionosphere": "Ionosphere",
    "iris": "Iris",
    "semeion": "Semeion",
    "sonar": "Sonar",
}

profile_plot_df["dataset_display"] = (
    profile_plot_df["dataset"]
    .map(dataset_display_names)
    .fillna(
        profile_plot_df["dataset"]
        .str.replace("_", " ")
        .str.title()
    )
)


dimension_columns = [
    "predictive_difficulty_score",
    "strategy_sensitivity_score",
    "computational_burden_score",
]

dimension_labels = [
    "Predictive\ndifficulty",
    "Strategy\nsensitivity",
    "Computational\nburden",
]

dimension_matrix = profile_plot_df[dimension_columns].to_numpy()


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

figure_height = max(5.5, 0.38 * len(profile_plot_df))

fig, ax = plt.subplots(
    figsize=(7.2, figure_height)
)

heatmap = ax.imshow(
    dimension_matrix,
    aspect="auto",
    vmin=0,
    vmax=1,
)

ax.set_xticks(np.arange(len(dimension_labels)))
ax.set_xticklabels(dimension_labels)

ax.set_yticks(np.arange(len(profile_plot_df)))
ax.set_yticklabels(profile_plot_df["dataset_display"])

ax.set_xlabel("Normalized characterization dimension")
ax.set_ylabel("Dataset")


# Add numerical values
for row_index in range(dimension_matrix.shape[0]):
    for column_index in range(dimension_matrix.shape[1]):
        value = dimension_matrix[row_index, column_index]

        if pd.notna(value):
            ax.text(
                column_index,
                row_index,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if value < 0.5 else "black",
            )


# Add horizontal separators between subsets
label_values = profile_plot_df["predictive_difficulty_label"].tolist()

for row_index in range(1, len(label_values)):
    if label_values[row_index] != label_values[row_index - 1]:
        ax.axhline(
            row_index - 0.5,
            linewidth=1.8,
            color="white",
        )


colorbar = fig.colorbar(
    heatmap,
    ax=ax,
    fraction=0.045,
    pad=0.04,
)

colorbar.set_label("Normalized score")

plt.tight_layout()

output_path = output_dir / "subset_dimension_profiles.pdf"

plt.savefig(
    output_path,
    bbox_inches="tight",
)

fig.savefig(output_dir / "subset_dimension_profiles.png", dpi=160, bbox_inches="tight")
plt.show()

print(f"Saved to: {output_path}")


## Exports and final consistency audit

All outputs below are isolated from the protected reference directories and historical files. The final audit checks the saved canonical profile, the 17-dataset pool, the article table, weights, seed outputs and file hashes. The targeted execution log lists exactly which dependency cells were refreshed; it does not claim a full notebook rerun.


In [ ]:
exports = {
    'target_audit.csv':target_audit_df, 'dataset_summary.csv':summary_df,
    'baseline_results.csv':baseline_df, 'baseline_cv_performance.csv':baseline_df,
    'baseline_characteristics.csv':baseline_characteristics_df,
    'baseline_dataset_characteristics.csv':baseline_characteristics_df,
    'baseline_cv_audit.csv':cv_audit_df,
    'computational_characteristics.csv':dataset_profile_df[['dataset',
        'baseline_runtime_seconds_mean','mean_tree_nodes','mean_tree_depth','mean_tree_leaves',
        'runtime_burden_score','tree_size_burden_score','computational_burden_score']],
    'dataset_experimental_profile.csv':dataset_profile_df,
    'predictive_difficulty_monotonicity_check.csv':predictive_monotonicity_df,
}
for filename,frame in exports.items():
    frame.to_csv(output_dir / filename,index=False)

checks=[]
def audit_check(name, condition):
    checks.append(dict(check=name, passed=bool(condition)))
    print(('PASS' if condition else 'FAIL')+': '+name)

p=dataset_profile_df
eligible=p.ranking_eligible
ranked_models=baseline_df.loc[baseline_df.dataset.isin(p.loc[eligible,'dataset'])
    & baseline_df.model.isin(NON_DUMMY_BASELINE_MODELS)]
audit_check('Every dataset has explicit target and ID configuration',set(datasets)<=set(TARGET_CONFIG) and set(datasets)<=set(ID_COLUMNS))
audit_check('All ranked datasets satisfy common CV eligibility',p.loc[eligible,'min_class_count'].ge(5).all() and p.loc[eligible,'comparable_cv'].all())
audit_check('All four non-dummy baselines completed',p.loc[eligible,'complete_non_dummy_protocol'].all()
    and ranked_models.groupby('dataset').model.nunique().eq(4).all())
audit_check('Every ranked model has 15 finite CV scores and no error',successful_rows(ranked_models).all())
model_aggregates_ok=True
for _, row in p.loc[eligible].iterrows():
    models=ranked_models.loc[ranked_models.dataset.eq(row.dataset)]
    means=np.array([np.mean(s) for s in models.scores])
    stds=np.array([np.std(s,ddof=1) for s in models.scores])
    model_aggregates_ok &= np.allclose([row.baseline_macro_f1_mean,row.cv_variability,row.strategy_sensitivity],
        [means.mean(),stds.mean(),means.max()-means.min()])
audit_check('F, V and S derive only from four non-dummy score vectors (sample std)',model_aggregates_ok)
audit_check('D_pred equals 0.70*z(E)+0.30*z(V)',np.allclose(p.predictive_difficulty_score,
    .70*p.predictive_error_score+.30*p.cv_variability_score,equal_nan=True))
audit_check('D_comp requires and averages both normalized components',np.allclose(p.computational_burden_score,
    (p.runtime_burden_score+p.tree_size_burden_score)/2,equal_nan=True))
for descriptor in ['strategy_sensitivity','computational_burden_score']:
    perturbed=p.copy(deep=True)
    perturbed[descriptor]=np.arange(len(p))*1000
    labels,ranks=assign_predictive_difficulty_groups(perturbed)
    audit_check(descriptor+' does not affect labels',labels.equals(p.predictive_difficulty_label)
        and ranks.equals(p.predictive_difficulty_rank))
audit_check('Mean F: Easy > Intermediate > Hard',is_monotonic_f1)
article_expected=p.loc[eligible].sort_values('predictive_difficulty_rank').predictive_error_score.round(4).to_numpy()
audit_check('Article table reports z(E), not raw E',np.array_equal(article_ranking_df['z(E_d)'].to_numpy(),article_expected))
audit_check('Downstream tables have required columns',set(['predictive_difficulty_rank','ranking_eligible',
    'ranking_exclusion_reason','cv_variability','possible_issue'])<=set(final_interpretation_df.columns))
audit_check('Canonical results unchanged by supporting analyses',p.equals(canonical_snapshot_df))
audit_check('Noneligible normalized quantities remain NaN',p.loc[~eligible,
    ['predictive_error_score','cv_variability_score','strategy_sensitivity_score',
     'predictive_difficulty_score','runtime_burden_score','tree_size_burden_score','computational_burden_score']].isna().all().all())
audit_check('Canonical alpha=0.70 has no subset changes',weight_sensitivity_summary_df.query('alpha == 0.70').n_subset_changes.eq(0).all())
audit_check('Loaded profile matches audited reference', alignment.assert_profile(p, alignment_context['reference']) is None)
audit_check('Protected artifacts remain unchanged', alignment.verify_protected() > 0)
final_consistency_audit_df=pd.DataFrame(checks)
final_consistency_audit_df.to_csv(output_dir / 'final_consistency_audit.csv',index=False)
failed=final_consistency_audit_df.loc[~final_consistency_audit_df.passed,'check'].tolist()
assert not failed, 'Consistency audit failed: '+'; '.join(failed)
print(f'All {len(checks)} consistency checks passed; outputs: {output_dir}')


## Supporting implementation

The following tagged cell contains the helper implementation used by this supplement. The setup cell loads these definitions in a separate namespace, preserving the analysis variables and allowing reuse across notebooks.

In [ ]:
# Loaded by the setup cell; definitions are isolated from analysis variables.
if __name__ != '__main__':
    """Read-only canonical audit and isolated sensitivity outputs.

    Run with the saved canonical environment; never invoke either protected notebook.
    Only this directory is writable by this workflow. Seed 42 is loaded, not refitted.
    """
    import ast
    import contextlib
    import hashlib
    import io
    import json
    import platform
    import subprocess
    import sys
    import time
    import warnings
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import scipy
    import sklearn
    from scipy.stats import spearmanr
    from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
    from sklearn.metrics import f1_score, make_scorer

    ROOT = Path(__file__).resolve().parents[2]
    HOME = Path(__file__).resolve().parent
    OUT = HOME / 'outputs'
    SOURCE = ROOT / 'results/new_characterization_experiments'
    TREE = ROOT / 'results/default_tree_strata'
    SEEDS = [7, 21, 42, 84, 123]
    ALPHAS = [.51, .60, .70, .80, .90]
    GROUPS = ['easy', 'intermediate', 'hard']
    NUMERIC = {'F': 'baseline_macro_f1_mean', 'E': 'predictive_error',
               'V': 'cv_variability', 'normalized_E': 'predictive_error_score',
               'normalized_V': 'cv_variability_score',
               'difficulty_score': 'predictive_difficulty_score',
               'position': 'predictive_difficulty_rank'}


    def sha(path):
        return hashlib.sha256(Path(path).read_bytes()).hexdigest()


    def write_json(path, obj):
        path = safe_path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix('.tmp')
        tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False), encoding='utf-8')
        tmp.replace(path)


    def safe_path(path):
        path = Path(path).resolve()
        if not path.is_relative_to(HOME.resolve()):
            raise ValueError(f'Write outside isolated auxiliary directory: {path}')
        return path


    def csv(frame, name):
        path = safe_path(OUT / name)
        path.parent.mkdir(parents=True, exist_ok=True)
        frame.to_csv(path, index=False)


    def protected_files():
        paths = [ROOT / 'experiments' / (n + '.ipynb') for n in
                 ['New_Characterization_Experiments', 'Default_Tree_Canonical_Strata']]
        for directory in ['results', 'Figures']:
            paths.extend(p for p in (ROOT / directory).rglob('*') if p.is_file())
        paths.extend(ROOT.glob('*.png'))
        return sorted(set(paths))


    def freeze_inputs():
        """Snapshot protected inputs for this execution, before analysis exports."""
        global _protected_inputs
        OUT.mkdir(parents=True, exist_ok=True)
        _protected_inputs = {str(p.relative_to(ROOT)): sha(p) for p in protected_files()}
        write_json(OUT / 'protected_inputs.json', _protected_inputs)


    def verify_protected():
        after = {str(p.relative_to(ROOT)): sha(p) for p in protected_files()}
        if _protected_inputs != after:
            raise ValueError('Protected file contents or inventory changed during this execution')
        return len(_protected_inputs)


    def read_models(path):
        frame = pd.read_csv(path, keep_default_na=False, float_precision='round_trip')
        for column in ['scores', 'tree_nodes', 'tree_depths', 'tree_leaves']:
            if column in frame:
                frame[column] = frame[column].map(json.loads)
        for column in ['f1_macro_mean', 'f1_macro_std', 'f1_macro_min', 'f1_macro_max',
                       'mean_tree_nodes', 'mean_tree_depth', 'mean_tree_leaves', 'runtime_seconds']:
            if column in frame:
                frame[column] = pd.to_numeric(frame[column], errors='coerce')
        return frame


    def keyed(frame, pool):
        if frame.dataset.duplicated().any() or set(frame.dataset) != set(pool):
            raise ValueError('Missing, extra, or duplicated datasets')
        return frame.set_index('dataset').loc[sorted(pool)]


    def assert_profile(frame, reference):
        frame = frame.loc[frame.ranking_eligible]
        a = keyed(frame, reference.dataset)
        b = keyed(reference, reference.dataset)
        for col in list(NUMERIC.values()):
            np.testing.assert_allclose(a[col], b[col], rtol=1e-12, atol=1e-14)
        if a.predictive_difficulty_label.tolist() != b.predictive_difficulty_label.tolist():
            raise ValueError('Canonical labels differ')


    def prepare():
        freeze_inputs()
        OUT.mkdir(exist_ok=True)
        sys.path.insert(0, str(ROOT))
        from experiments.default_tree_strata import canonical_components
        from experiments.characterization_persistence import code_digest, data_digest, validate_evaluations
        with contextlib.redirect_stdout(io.StringIO()):
            ns, notebook = canonical_components(ROOT)
        metadata = json.loads((SOURCE / 'metadata.json').read_text())
        protocol = metadata['protocol']
        tree_meta = json.loads((TREE / 'metadata.json').read_text())
        from importlib.metadata import version
        versions = {k: version(k if k != 'sklearn' else 'scikit-learn')
                    for k in protocol['versions'] if k != 'python'}
        versions['python'] = platform.python_version()
        if versions != protocol['versions']:
            raise ValueError(f'Use canonical Python environment; expected {protocol["versions"]}; got {versions}')
        if protocol['cv'] != {'folds': 5, 'repeats': 3, 'seed': 42}:
            raise ValueError('Changed canonical CV protocol')
        if protocol['scoring'] != 'macro_f1; zero_division=0' or protocol['std_ddof'] != 1:
            raise ValueError('Changed scorer or variability convention')
        for key, attr in [('target_config', 'TARGET_CONFIG'), ('id_columns', 'ID_COLUMNS'),
                          ('missing_tokens', 'MISSING_TOKENS')]:
            if protocol[key] != ns[attr]:
                raise ValueError(f'Configuration mismatch: {key}')
        for name, expected in protocol['code'].items():
            if code_digest(ns[name]) != expected:
                raise ValueError(f'Source evaluation definition differs: {name}')
        for name, expected in tree_meta['source_sha256'].items():
            if sha(SOURCE / name) != expected:
                raise ValueError(f'Two reference provenances differ: {name}')
        model_configs = json.loads((SOURCE / 'canonical_model_configuration.json').read_text())
        if model_configs != ns['MODEL_CONFIGS']:
            raise ValueError('Model configurations differ')
        manifest = json.loads((SOURCE / 'canonical_pool.json').read_text())
        pool = manifest['eligible']
        if len(pool) != 17 or len(set(pool)) != 17 or set(ns['loaded']) != set(manifest['selected_datasets']):
            raise ValueError('Invalid canonical pool')
        for name, obj in ns['loaded'].items():
            identity = metadata['datasets'][name]
            if sha(ns['datasets'][name]) != identity['file_sha256'] or data_digest(obj) != identity['filtered_data_sha256']:
                raise ValueError(f'Data identity differs: {name}')
        baseline = read_models(SOURCE / 'tables/baseline_results.csv')
        validate_evaluations(baseline, pool, ns['NON_DUMMY_BASELINE_MODELS'])
        audit = pd.read_csv(SOURCE / 'tables/baseline_cv_audit.csv')
        aggregated = ns['aggregate_baselines'](baseline, audit)
        profile = pd.read_csv(SOURCE / 'tables/dataset_experimental_profile.csv', float_precision='round_trip')
        canonical = pd.read_csv(SOURCE / 'tables/ablation_long.csv', float_precision='round_trip')
        canonical = canonical.loc[canonical.variant.eq('canonical')].copy()
        tree_reference = pd.read_csv(TREE / 'tables/canonical_reference.csv', float_precision='round_trip')
        pd.testing.assert_frame_equal(keyed(canonical, pool), keyed(tree_reference, pool),
                                      check_exact=False, rtol=1e-12, atol=1e-14)
        reference = profile.loc[profile.ranking_eligible].copy()
        a, b = keyed(canonical, pool), keyed(reference, pool)
        tree_keyed = keyed(tree_reference, pool)
        checks = [dict(reference='default_tree_saved', field=col,
                       max_absolute_difference=float((a[col]-tree_keyed[col]).abs().max()), passed=True)
                  for col in NUMERIC]
        for left, right in NUMERIC.items():
            delta = float((a[left] - b[right]).abs().max())
            np.testing.assert_allclose(a[left], b[right], rtol=1e-12, atol=1e-14)
            checks.append(dict(reference='new_experimental_profile', field=left, max_absolute_difference=delta, passed=True))
        if a.difficulty_group.tolist() != b.predictive_difficulty_label.tolist():
            raise ValueError('Reference label mismatch')
        if a.difficulty_group.value_counts().reindex(GROUPS).tolist() != [6, 6, 5]:
            raise ValueError('Invalid canonical strata counts')
        reconstructed = score_profile(aggregated.loc[aggregated.ranking_eligible], ns)
        assert_profile(reconstructed, reference)
        # Verify saved partition identities and every original non-dummy fold score.
        partitions = pd.read_csv(SOURCE / 'tables/cv_partitions.csv')
        folds = pd.read_csv(SOURCE / 'tables/baseline_fold_scores.csv', float_precision='round_trip')
        fold_index = folds.set_index(['dataset', 'model', 'repeat', 'fold'], verify_integrity=True)
        for name in pool:
            obj = ns['loaded'][name]
            rows = partitions.loc[partitions.dataset.eq(name)].sort_values(['repeat', 'fold'])
            splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42).split(obj['X'], obj['y']))
            if list(zip(rows.repeat, rows.fold)) != [(i//5+1, i%5+1) for i in range(15)]:
                raise ValueError(f'Invalid saved partitions: {name}')
            for i, (row, (train, test)) in enumerate(zip(rows.itertuples(), splits)):
                np.testing.assert_array_equal(json.loads(row.train_indices), train)
                np.testing.assert_array_equal(json.loads(row.test_indices), test)
                split_hash = hashlib.sha256(train.tobytes() + test.tobytes()).hexdigest()
                if row.split_sha256 != split_hash or row.data_sha256 != data_digest(obj):
                    raise ValueError('Partition identity mismatch')
                for model in ns['NON_DUMMY_BASELINE_MODELS']:
                    saved = fold_index.loc[(name, model, row.repeat, row.fold)]
                    score = baseline.loc[baseline.dataset.eq(name) & baseline.model.eq(model), 'scores'].iloc[0][i]
                    if saved.split_sha256 != split_hash or saved.data_sha256 != data_digest(obj):
                        raise ValueError('Fold provenance mismatch')
                    np.testing.assert_allclose(saved.macro_f1, score, rtol=1e-12, atol=1e-14)
        csv(pd.DataFrame(checks), 'reference_audit.csv')
        if (ROOT / 'results/dataset_statistics/dataset_experimental_profile.csv').exists():
            old = pd.read_csv(ROOT / 'results/dataset_statistics/dataset_experimental_profile.csv')
            diff = reference.merge(old, on='dataset', suffixes=('_canonical', '_old'), validate='one_to_one')
            columns = ['dataset']
            for name in NUMERIC.values():
                diff[name + '_delta'] = diff[name + '_canonical'] - diff[name + '_old']
                columns += [name + '_old', name + '_canonical', name + '_delta']
            diff['label_changed'] = diff.predictive_difficulty_label_canonical.ne(diff.predictive_difficulty_label_old)
            csv(diff[columns + ['predictive_difficulty_label_old', 'predictive_difficulty_label_canonical', 'label_changed']],
                'historical_vs_canonical.csv')
        notebook_history_note = ('Publication source edition: whole-notebook hashes can differ after '
                                 'editorial cleanup. Evaluation function identities and saved artifact '
                                 'hashes are verified above; no Git history is required.')
        notebook_history_verified = False
        provenance = dict(canonical_source=str(SOURCE / 'tables/dataset_experimental_profile.csv'),
                          canonical_sha256=sha(SOURCE / 'tables/dataset_experimental_profile.csv'),
                          original_protocol=protocol, tree_source_hashes_verified=True,
                          current_reference_notebook_sha256=sha(notebook),
                          tree_recorded_reference_notebook_sha256=tree_meta['canonical_notebook_sha256'],
                          reference_notebook_hash_matches=sha(notebook)==tree_meta['canonical_notebook_sha256'],
                          evaluation_function_hashes_match=True,
                          notebook_hash_note=notebook_history_note,
                          notebook_history_verified_against_git=notebook_history_verified,
                          historical_seed_reuse=False,
                          historical_seed_reason='Historical metadata: Python 3.12.12, NumPy 2.4.1, SciPy 1.17.0, sklearn 1.8.0; canonical: Python 3.11.5, NumPy 1.23.5, SciPy 1.15.3, sklearn 1.3.0. Seed logs explicitly show sklearn 1.8 warnings; no per-seed signed configuration/partition manifest.',
                          seed42_status='Loaded saved four-model results; algebraically verified from folds; NOT reproduced by new training.',
                          protected_file_count=verify_protected())
        write_json(OUT / 'provenance.json', provenance)
        return dict(ns=ns, baseline=baseline, audit=audit, profile=profile, reference=reference,
                    pool=pool, metadata=metadata, provenance=provenance)


    def score_profile(aggregated, ns, alpha=.70):
        result = aggregated.copy(deep=True)
        result['predictive_error'] = 1 - result.baseline_macro_f1_mean
        norm = ns['minmax_normalize_on_pool']
        result['predictive_error_score'] = norm(result.predictive_error, result.ranking_eligible)
        result['cv_variability_score'] = norm(result.cv_variability, result.ranking_eligible)
        result['predictive_difficulty_score'] = alpha * result.predictive_error_score + (.30 if alpha == .70 else 1-alpha) * result.cv_variability_score
        labels, ranks = ns['assign_predictive_difficulty_groups'](result)
        result['predictive_difficulty_label'], result['predictive_difficulty_rank'] = labels, ranks
        return result


    def compare(candidate, reference, parameter, value):
        a, b = keyed(candidate, reference.dataset), keyed(reference, reference.dataset)
        detail = pd.DataFrame(dict(dataset=a.index, canonical_rank=b.predictive_difficulty_rank,
            alternative_rank=a.predictive_difficulty_rank, canonical_subset=b.predictive_difficulty_label,
            alternative_subset=a.predictive_difficulty_label,
            canonical_score=b.predictive_difficulty_score, alternative_score=a.predictive_difficulty_score)).reset_index(drop=True)
        detail[parameter] = value
        detail['rank_difference'] = detail.alternative_rank - detail.canonical_rank
        detail['subset_changed'] = detail.alternative_subset.ne(detail.canonical_subset)
        rho = spearmanr(detail.canonical_rank, detail.alternative_rank).statistic
        summary = {parameter:value, 'n_datasets':len(detail), 'spearman_rank_correlation':float(rho),
                   'n_subset_changes':int(detail.subset_changed.sum()),
                   'pct_subset_changes':100*float(detail.subset_changed.mean()),
                   'label_agreement':1-float(detail.subset_changed.mean()),
                   'percent_same_label':100*(1-float(detail.subset_changed.mean())),
                   'mean_absolute_rank_change':float(detail.rank_difference.abs().mean()),
                   'max_absolute_rank_change':float(detail.rank_difference.abs().max()),
                   'datasets_crossing_boundaries':'; '.join(detail.loc[detail.subset_changed, 'dataset'])}
        return summary, detail


    def weights(context):
        rows, details, profiles = [], [], []
        for alpha in ALPHAS:
            candidate = context['reference'].copy(deep=True)
            candidate['predictive_difficulty_score'] = (alpha*candidate.predictive_error_score
                + (.30 if alpha == .70 else 1-alpha)*candidate.cv_variability_score)
            labels, ranks = context['ns']['assign_predictive_difficulty_groups'](candidate)
            candidate['predictive_difficulty_label'], candidate['predictive_difficulty_rank'] = labels, ranks
            if alpha == .70:
                assert_profile(candidate, context['reference'])
            summary, detail = compare(candidate, context['reference'], 'alpha', alpha)
            rows.append(summary); details.append(detail)
            candidate['alpha'] = alpha; profiles.append(candidate)
        summary, detail = pd.DataFrame(rows), pd.concat(details, ignore_index=True)
        csv(summary, 'weight_sensitivity_summary.csv'); csv(detail, 'weight_sensitivity_by_dataset.csv')
        csv(pd.concat(profiles, ignore_index=True), 'weight_sensitivity_profiles.csv')
        return summary, detail


    def seed_models(context, seed, allow_fit=False):
        ns = context['ns']
        if seed == 42:
            return context['baseline'].loc[context['baseline'].dataset.isin(context['pool']) &
                context['baseline'].model.isin(ns['NON_DUMMY_BASELINE_MODELS'])].copy(deep=True)
        from experiments.characterization_persistence import data_digest, code_digest
        rows, partition_rows = [], []
        for dataset in sorted(context['pool']):
            obj = ns['loaded'][dataset]
            splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=seed).split(obj['X'], obj['y']))
            split_hashes = [hashlib.sha256(a.tobytes()+b.tobytes()).hexdigest() for a,b in splits]
            for i, ((train, test), h) in enumerate(zip(splits, split_hashes)):
                partition_rows.append(dict(seed=seed, dataset=dataset, repeat=i//5+1, fold=i%5+1,
                    train_indices=json.dumps(train.tolist()), test_indices=json.dumps(test.tolist()),
                    data_sha256=data_digest(obj), split_sha256=h))
            for model in ns['NON_DUMMY_BASELINE_MODELS']:
                estimator = ns['make_baseline'](model, obj['X'])
                spec = dict(canonical_protocol=context['metadata']['protocol'], cv_seed=seed,
                            data=context['metadata']['datasets'][dataset], split_hashes=split_hashes,
                            model=model, params={k:repr(v) for k,v in estimator.get_params(deep=True).items()},
                            n_jobs=1, evaluator_sha256=code_digest(seed_models))
                signature = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()
                path = OUT / 'seed_cache' / f'{seed}__{dataset}__{model}.json'
                if path.exists():
                    payload = json.loads(path.read_text())
                    if payload['signature'] != signature or payload['specification'] != spec:
                        raise ValueError(f'Incompatible seed checkpoint: {path}')
                    row = payload['row']
                else:
                    if not allow_fit:
                        raise FileNotFoundError(f'Missing verified seed evaluation: {path}. Run workflow.py --fit-seeds in canonical environment.')
                    print(f'Fitting seed={seed} {dataset}/{model}', flush=True)
                    started = time.perf_counter()
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter('always')
                        result = cross_validate(estimator, obj['X'], obj['y'], cv=splits,
                            scoring=make_scorer(f1_score, average='macro', zero_division=0),
                            n_jobs=1, error_score='raise')
                    scores = result['test_score']
                    row = dict(dataset=dataset, model=model, cv_seed=seed, cv_folds=5, cv_repeats=3,
                        n_scores=15, scores=scores.tolist(), error='',
                        warnings='\n'.join(sorted({f'{w.category.__name__}: {w.message}' for w in caught})),
                        f1_macro_mean=float(scores.mean()), f1_macro_std=float(scores.std(ddof=1)),
                        f1_macro_min=float(scores.min()), f1_macro_max=float(scores.max()),
                        runtime_seconds=time.perf_counter()-started)
                    write_json(path, dict(signature=signature, specification=spec, row=row))
                scores = np.asarray(row['scores'])
                if row['cv_seed'] != seed or row['cv_folds'] != 5 or row['cv_repeats'] != 3 or row['error'] or row['n_scores'] != 15 or scores.shape != (15,) or not np.isfinite(scores).all():
                    raise ValueError('Invalid seed evaluation')
                np.testing.assert_allclose([row['f1_macro_mean'], row['f1_macro_std']], [scores.mean(), scores.std(ddof=1)], rtol=1e-12, atol=1e-14)
                rows.append(row)
        csv(pd.DataFrame(partition_rows), f'seed_partitions_{seed}.csv')
        return pd.DataFrame(rows)


    def seeds(context, allow_fit=False):
        models, profiles, summaries, details, fold_rows = [], [], [], [], []
        for seed in SEEDS:
            frame = seed_models(context, seed, allow_fit)
            expected = {(d,m) for d in context['pool'] for m in context['ns']['NON_DUMMY_BASELINE_MODELS']}
            if frame.duplicated(['dataset','model']).any() or set(zip(frame.dataset,frame.model)) != expected:
                raise ValueError('Seed is incomplete; never normalize a smaller pool')
            agg = frame.groupby('dataset', as_index=False).agg(baseline_macro_f1_mean=('f1_macro_mean','mean'), cv_variability=('f1_macro_std','mean'))
            agg['ranking_eligible'] = True
            profile = score_profile(agg, context['ns'])
            if seed == 42:
                assert_profile(profile, context['reference'])
                # Adopt the exact saved canonical representation after algebraic check.
                profile = context['reference'].copy(deep=True)
            profile['seed'] = seed
            profile['normalization_pool_size'] = len(context['pool'])
            for col in ['predictive_error','cv_variability']:
                profile[col+'_pool_min'] = profile[col].min()
                profile[col+'_pool_max'] = profile[col].max()
            summary, detail = compare(profile, context['reference'], 'seed', seed)
            summary['reference_status'] = 'loaded_not_refitted' if seed == 42 else 'evaluated_canonical_protocol'
            summaries.append(summary); details.append(detail); profiles.append(profile)
            frame['seed'] = seed; models.append(frame)
            for r in frame.itertuples():
                for i, score in enumerate(r.scores):
                    fold_rows.append(dict(seed=seed, dataset=r.dataset, model=r.model, repeat=i//5+1, fold=i%5+1, macro_f1=score))
        summary, detail, scores = pd.DataFrame(summaries), pd.concat(details, ignore_index=True), pd.concat(profiles, ignore_index=True)
        stability = detail.groupby('dataset', as_index=False).agg(canonical_rank=('canonical_rank','first'),
            canonical_subset=('canonical_subset','first'), mean_rank=('alternative_rank','mean'), std_rank=('alternative_rank','std'),
            min_rank=('alternative_rank','min'), max_rank=('alternative_rank','max'), n_label_changes=('subset_changed','sum'),
            proportion_same_label=('subset_changed',lambda s:1-s.mean()),
            max_absolute_rank_change=('rank_difference',lambda s:s.abs().max()),
            labels_observed=('alternative_subset',lambda s:', '.join(g for g in GROUPS if g in set(s))))
        score_stats = scores.groupby('dataset', as_index=False).agg(mean_D_pred=('predictive_difficulty_score','mean'), std_D_pred=('predictive_difficulty_score','std'))
        stability = stability.merge(score_stats, on='dataset', validate='one_to_one').sort_values('canonical_rank')
        exports = {'seed_robustness_summary.csv':summary, 'seed_robustness_comparison_vs_canonical.csv':detail,
                   'seed_robustness_dataset_scores.csv':scores, 'seed_robustness_dataset_summary.csv':stability,
                   'seed_robustness_model_results.csv':pd.concat(models, ignore_index=True),
                   'seed_robustness_fold_scores.csv':pd.DataFrame(fold_rows)}
        for name, frame in exports.items():
            csv(frame, name)
        return summary, detail, stability, scores


    def figures(weight_summary, seed_summary, detail, stability, scores):
        import matplotlib.pyplot as plt
        def save(fig, name):
            fig.tight_layout()
            for ext in ['png','pdf']:
                fig.savefig(safe_path(OUT / f'{name}.{ext}'), dpi=160, bbox_inches='tight')
            plt.close(fig)
        fig, axes = plt.subplots(1,2,figsize=(11,4))
        axes[0].plot(weight_summary.alpha, weight_summary.spearman_rank_correlation, 'o-')
        axes[0].set(xlabel='Predictive error weight (alpha)', ylabel='Spearman vs canonical ranking', ylim=(.8,1.01))
        axes[1].plot(weight_summary.alpha, weight_summary.n_subset_changes, 'o-')
        axes[1].set(xlabel='Predictive error weight (alpha)', ylabel='Datasets changing stratum', yticks=range(0,8))
        save(fig,'weight_sensitivity')
        names = stability.dataset.tolist()
        fig, ax = plt.subplots(figsize=(11,7))
        palette = plt.get_cmap('tab20')(np.linspace(0, 1, len(names)))
        for color, name in zip(palette, names):
            d = detail.loc[detail.dataset.eq(name)].set_index('seed').loc[SEEDS]
            ax.plot(range(5), d.alternative_rank, marker='o', label=name, color=color)
        ax.axhline(6.5,color='gray',linestyle='--'); ax.axhline(12.5,color='gray',linestyle='--')
        ax.set(xticks=range(5),xticklabels=['7','21','42 (saved)','84','123'],yticks=range(1,18),
               xlabel='CV partition seed',ylabel='Difficulty rank',ylim=(17.5,.5))
        ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=8)
        save(fig,'cv_seed_rank_stability')
        fig, ax = plt.subplots(figsize=(10,6))
        colors = {'easy':'#009E73','intermediate':'#0072B2','hard':'#D55E00'}
        matrix = detail.pivot(index='dataset',columns='seed',values='alternative_subset').loc[names,SEEDS]
        for i,name in enumerate(names):
            for j,seed in enumerate(SEEDS):
                label=matrix.loc[name,seed]; ax.scatter(j,i,c=colors[label],s=320,marker='s')
        ax.set(yticks=range(17),yticklabels=names,xticks=range(5),xticklabels=['7','21','42 (saved)','84','123'],
               xlabel='CV partition seed',ylim=(16.8,-.8),title='Green: Easy | Blue: Intermediate | Orange: Hard')
        save(fig,'cv_seed_label_stability')
        fig, ax=plt.subplots(figsize=(11,5))
        values=[scores.loc[scores.dataset.eq(n),'predictive_difficulty_score'].to_numpy() for n in names]
        ax.boxplot(values,labels=names)
        ax.scatter(range(1,18), scores.loc[scores.seed.eq(42)].set_index('dataset').loc[names,'predictive_difficulty_score'],c='red',s=20,label='Saved seed 42')
        ax.tick_params(axis='x',rotation=65); ax.set_ylabel('Predictive difficulty score');ax.legend()
        save(fig,'cv_seed_predictive_difficulty_stability')
